In [1]:
Encoding = {"PAD":0, "a": 1, "b": 2, "c": 3, "d": 4, "e": 5, "f": 6, "g": 7, "h": 8, "i": 9, "j": 10, "k": 11, "l": 12, "m": 13, "n": 14, "o": 15, "p": 16, "q": 17, "r": 18, "s": 19, "t": 20, "u": 21, "v": 22, "w": 23, "x": 24, "y": 25, "z": 26, "<": 27, ">": 28, " ": 29, }
Decoding = {0: "PAD", 1: "a", 2: "b", 3: "c", 4: "d", 5: "e", 6: "f", 7: "g", 8: "h", 9: "i", 10: "j", 11: "k", 12: "l", 13: "m", 14: "n", 15: "o", 16: "p", 17: "q", 18: "r", 19: "s", 20: "t", 21: "u", 22: "v", 23: "w", 24: "x", 25: "y", 26: "z", 27: "<", 28: ">", 29: " ", 30: ""}

In [2]:
def Encoder(sentence):
  Fillup = 300 - len(sentence)
  encoded_chars = [Encoding[char] for char in sentence if char in Encoding]
  return [27] + encoded_chars + [28] + [30] * (300 - len(encoded_chars))

def EncoderForInference(sentence):
  encoded_chars = [Encoding[char] for char in sentence if char in Encoding]
  return encoded_chars

def Decoder(tokens):
  return "".join([Decoding[token] for token in tokens])

# Token Positional encoding

In [3]:
import torch
import torch.nn as nn

class EmbeddingLayer(nn.Module):
    def __init__(self, vocab_size=31, d_model=64, max_seq_len=302):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size,d_model)
        self.position_embedding = nn.Embedding(max_seq_len,d_model)

    def forward(self, x):
        batch_size, seq_len = x.shape
        positions = torch.arange(seq_len,device=x.device).unsqueeze(0)

        token_emb = self.token_embedding(x)
        pos_emb = self.position_embedding(positions)

        return token_emb + pos_emb

# MHA

In [4]:
class MultiHeadAttention(nn.Module):

    def __init__(self,d_model=64,num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)

        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x, mask):
        B, T, C = x.shape

        Q = self.q_proj(x)
        K = self.k_proj(x)
        V = self.v_proj(x)

        Q = Q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        scores = (Q @ K.transpose(-2, -1))
        scores = scores / (self.head_dim ** 0.5)
        scores = scores.masked_fill(mask == 0,float('-inf'))

        attention = torch.softmax(scores, dim=-1)

        out = attention @ V
        out = out.transpose(1, 2).contiguous()
        out = out.view(B, T, C)

        return self.out_proj(out)

# Feed Forward

In [5]:
class FeedForward(nn.Module):

    def __init__(self,d_model=64,hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, d_model)
        )

    def forward(self, x):
        return self.net(x)

# Decoder Block

In [6]:
class DecoderBlock(nn.Module):

    def __init__(self, d_model=128, num_heads=8, hidden_dim=256):
        super().__init__()
        self.attn1 = MultiHeadAttention(d_model,num_heads)
        self.attn2 = MultiHeadAttention(d_model,num_heads)

        self.ffn = FeedForward(d_model,hidden_dim)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

    def forward(self, x, mask):
        x = x + self.attn1(self.norm1(x),mask)
        x = x + self.attn2(self.norm2(x),mask)
        x = x + self.ffn(self.norm3(x))

        return x

# Full Decoder

In [7]:
class CharacterTransformer(nn.Module):

    def __init__(self, vocab_size=31, d_model=256, max_seq_len=302, num_layers=6, num_heads=16, hidden_dim=512):
        super().__init__()
        self.embedding = EmbeddingLayer(vocab_size,d_model,max_seq_len)
        self.blocks = nn.ModuleList([
            DecoderBlock(d_model,num_heads,hidden_dim)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.classifier = nn.Linear(d_model,vocab_size)

    def causal_mask(self, seq_len, device):
        mask = torch.tril(torch.ones(seq_len, seq_len,device=device))
        return mask.unsqueeze(0).unsqueeze(0)

    def forward(self, x):
        B, T = x.shape
        mask = self.causal_mask(T,x.device)

        x = self.embedding(x)
        for block in self.blocks:
            x = block(x, mask)

        x = self.norm(x)
        logits = self.classifier(x)

        return logits

# import BookCorpus `SamuelYang/bokcorpus` Clean data

In [8]:
!pip install gdown
import gdown

file_id = "10huFLWk3dwL4hmWB6u5o26EPah_teygP"

gdown.download(
    f"https://drive.google.com/uc?id={file_id}",
    "bookcorpus_clean.csv",
    quiet=False
)

import pandas as pd

data = pd.read_csv("bookcorpus_clean.csv", nrows=1_500_000)
data.head()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [gdown]


Downloading...
From (original): https://drive.google.com/uc?id=10huFLWk3dwL4hmWB6u5o26EPah_teygP
From (redirected): https://drive.google.com/uc?id=10huFLWk3dwL4hmWB6u5o26EPah_teygP&confirm=t&uuid=5506f24d-650e-4b9c-8ff1-5a55cf7070bf
To: /teamspace/studios/this_studio/bookcorpus_clean.csv
100%|██████████| 3.12G/3.12G [00:20<00:00, 154MB/s] 


,text
0,the halfling book one in the fall of igneeria ...
1,isbn isbn for my family who encouraged me to n...
2,starlings new york is not the place youd expec...
3,its a small quiet town the kind where everyone...
4,its a place where your parents wouldnt even ca...


# Encode Data To Numbers

In [9]:
NumData = torch.tensor(
    [Encoder(sentence) for sentence in data["text"]],
    dtype=torch.long
)

print("Encoded Data Shape: ", NumData.shape)
print("SampleEncoded Data: ", NumData[:2])

Encoded Data Shape:  torch.Size([1500000, 302])
SampleEncoded Data:  tensor([[27, 20,  8,  5, 29,  8,  1, 12,  6, 12,  9, 14,  7, 29,  2, 15, 15, 11,
         29, 15, 14,  5, 29,  9, 14, 29, 20,  8,  5, 29,  6,  1, 12, 12, 29, 15,
          6, 29,  9,  7, 14,  5,  5, 18,  9,  1, 29, 19,  5, 18,  9,  5, 19, 29,
         11,  1, 25, 12,  5,  5, 29, 19, 15,  4,  5, 18,  2, 21, 18,  7, 29,  3,
         15, 16, 25, 18,  9,  7,  8, 20, 29, 11,  1, 25, 12,  5,  5, 29, 19, 15,
          4,  5, 18,  2, 21, 18,  7, 29,  1, 12, 12, 29, 18,  9,  7,  8, 20, 19,
         29, 18,  5, 19,  5, 18, 22,  5,  4, 28, 30, 30, 30, 30, 30, 30, 30, 30,
         30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30,
         30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30,
         30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30,
         30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30,
         30, 30, 30, 30, 30, 30, 30, 30,

In [10]:
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

X = NumData[:, :-1]
Y = NumData[:, 1:]

train_dataset = TensorDataset(X,Y)

train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True
)

# Training loop

In [11]:
device = "cuda"
model = CharacterTransformer().to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2.5e-4
)

criterion = nn.CrossEntropyLoss()

total = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total:,}")


Total parameters: 4,838,431


In [ ]:
import time
import torch
from torch.amp import autocast, GradScaler
from datetime import datetime
from zoneinfo import ZoneInfo

print("Started at:", datetime.now(ZoneInfo("Asia/Kolkata")).strftime("%a %b %d %H:%M:%S %Y")) # Mon Jun 15 14:45:19 2026

scaler = GradScaler("cuda")

epochs = 5

for epoch in range(epochs):
    print(f"Epoch {epoch + 1}/{epochs}")
    model.train()

    for i, (x_batch, y_batch) in enumerate(train_loader):

        x_batch = x_batch.to(device, dtype=torch.long)
        y_batch = y_batch.to(device, dtype=torch.long)

        optimizer.zero_grad(set_to_none=True)

        with autocast(device_type="cuda", dtype=torch.float16):
            logits = model(x_batch)

            loss = criterion(
                logits.reshape(-1, 31),
                y_batch.reshape(-1)
            )

        # Backward pass
        scaler.scale(loss).backward()

        # Update weights
        scaler.step(optimizer)
        scaler.update()

        if i % 1000 == 0 and i != 0:
            print(f"First {i} batches processed...")

    print(f"Loss: {loss.item():.4f}\n")

print("Ended at:", datetime.now(ZoneInfo("Asia/Kolkata")).strftime("%a %b %d %H:%M:%S %Y"))

Started at: Mon Jun 15 09:15:06 2026
Epoch 1/5
First 1000 batches processed...
First 2000 batches processed...
First 3000 batches processed...
First 4000 batches processed...
First 5000 batches processed...
Loss: 0.3900

Epoch 2/5
First 1000 batches processed...
First 2000 batches processed...
First 3000 batches processed...
First 4000 batches processed...
First 5000 batches processed...
Loss: 0.3879

Epoch 3/5
First 1000 batches processed...
First 2000 batches processed...
First 3000 batches processed...
First 4000 batches processed...
First 5000 batches processed...
Loss: 0.3516

Epoch 4/5
First 1000 batches processed...
First 3000 batches processed...
First 4000 batches processed...


In [ ]:
torch.save(model.state_dict(), "character_transformer.pth")
print("Model saved!")

# Inference loop

In [ ]:
import re

@torch.no_grad()
def generate(model, startChar, max_new_tokens=50, temperature=0.8):

    model.eval()

    tokens = torch.tensor(
        [EncoderForInference(re.sub(r'[^A-Za-z\s]', '', startChar.lower()))],
        dtype=torch.long
    ).cuda()
    startChar="< "+ startChar
    print(startChar, end="", flush=True)
    for _ in range(max_new_tokens):
        tokens_input = tokens[:, -301:]
        logits = model(tokens_input)
        next_token_logits = logits[:, -1]
        probs = torch.softmax(next_token_logits, dim=-1)
        logits_scaled = next_token_logits / temperature
        probs = torch.softmax(logits_scaled, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1).squeeze(1)
        tokens = torch.cat(
            [tokens, next_token.unsqueeze(1)],
            dim=1
        )

        # print each character live as it's predicted
        print(Decoding[next_token.item()], end="", flush=True)

In [ ]:
generate(model, "cryvutiuyt90", max_new_tokens=302)